# Part 3 – Advanced Modeling
This notebook implements the ensemble learning, tuning, pipeline, feature ablation, cross-validation, learning curve, and model serialization tasks. It expects `cleaned_data.csv` in the same folder.

In [ ]:

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt

df=pd.read_csv("cleaned_data.csv")

target="Sales" if "Sales" in df.columns else df.select_dtypes(include="number").columns[-1]
y=(df[target]>df[target].median()).astype(int)
X=pd.get_dummies(df.drop(columns=[target]),drop_first=True)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

# Decision Trees
tree=DecisionTreeClassifier(random_state=42)
tree.fit(X_train_s,y_train)
print("Default DT Train/Test:",
accuracy_score(y_train,tree.predict(X_train_s)),
accuracy_score(y_test,tree.predict(X_test_s)))

tree2=DecisionTreeClassifier(max_depth=5,min_samples_split=20,random_state=42)
tree2.fit(X_train_s,y_train)
print("Controlled DT Train/Test:",
accuracy_score(y_train,tree2.predict(X_train_s)),
accuracy_score(y_test,tree2.predict(X_test_s)))

for crit in ["gini","entropy"]:
    m=DecisionTreeClassifier(max_depth=5,criterion=crit,random_state=42)
    m.fit(X_train_s,y_train)
    print(crit,accuracy_score(y_test,m.predict(X_test_s)))

# Random Forest
rf=RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)
rf.fit(X_train_s,y_train)
rf_auc=roc_auc_score(y_test,rf.predict_proba(X_test_s)[:,1])
print("RF Train/Test/AUC",
accuracy_score(y_train,rf.predict(X_train_s)),
accuracy_score(y_test,rf.predict(X_test_s)),
rf_auc)

imp=pd.DataFrame({"Feature":X.columns,"Importance":rf.feature_importances_})
print(imp.sort_values("Importance",ascending=False).head())

low5=imp.sort_values("Importance").head(5)["Feature"].tolist()
Xtr_red=X_train.drop(columns=low5)
Xte_red=X_test.drop(columns=low5)

sc2=StandardScaler()
Xtr_red=sc2.fit_transform(Xtr_red)
Xte_red=sc2.transform(Xte_red)

rf2=RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)
rf2.fit(Xtr_red,y_train)
print("Reduced RF AUC:",roc_auc_score(y_test,rf2.predict_proba(Xte_red)[:,1]))

# Gradient Boosting
gb=GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,max_depth=3,random_state=42)
gb.fit(X_train_s,y_train)
print("GB AUC:",roc_auc_score(y_test,gb.predict_proba(X_test_s)[:,1]))

# Cross Validation
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
models={
"Logistic":LogisticRegression(max_iter=1000),
"DecisionTree":DecisionTreeClassifier(max_depth=5,min_samples_split=20,random_state=42),
"RandomForest":RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42),
"GradientBoosting":GradientBoostingClassifier(random_state=42)
}
for name,model in models.items():
    pipe=make_pipeline(StandardScaler(),model)
    s=cross_val_score(pipe,X,y,cv=cv,scoring="roc_auc")
    print(name,s.mean(),s.std())

pipe=make_pipeline(SimpleImputer(strategy="median"),
                   StandardScaler(),
                   RandomForestClassifier(random_state=42))

grid={
'randomforestclassifier__n_estimators':[50,100,200],
'randomforestclassifier__max_depth':[5,10,None],
'randomforestclassifier__min_samples_leaf':[1,5]
}

gs=GridSearchCV(pipe,grid,cv=cv,scoring="roc_auc",n_jobs=-1)
gs.fit(X_train,y_train)
print(gs.best_params_)
print(gs.best_score_)

rows=[]
for f in [0.2,0.4,0.6,0.8,1.0]:
    n=int(f*len(X_train))
    bp=gs.best_estimator_
    bp.fit(X_train.iloc[:n],y_train.iloc[:n])
    tr=roc_auc_score(y_train.iloc[:n],bp.predict_proba(X_train.iloc[:n])[:,1])
    te=roc_auc_score(y_test,bp.predict_proba(X_test)[:,1])
    rows.append([f,tr,te])
print(pd.DataFrame(rows,columns=["Fraction","Train AUC","Test AUC"]))

joblib.dump(gs.best_estimator_,"best_model.pkl")
loaded=joblib.load("best_model.pkl")
print(loaded.predict(X_test.iloc[:2]))
